# 4. 调用模型
在 LangChain 中，模型调用（Invocation）是指通过特定方法触发大语言模型生成输出的过程。根据不
同的应用场景和需求，LangChain 提供了几种核心的调用方式，主要是 invoke() 、 stream() 和batch() 方法，以及它们的异步版本 ainvoke() 、 astream() 和 abatch() ，下面将系统地介绍这些方
法。

* invoke() ：阻塞式，一次性返回完整结果问答、批处理任务、无需实时反馈的场景。
* ainvoke() ：非阻塞式，提高系统吞吐量高并发Web应用、IO密集型任务。
* stream() ：流式输出，实时返回每个token聊天机器人、长文本生成、需要提升用户体验的交互
应用。
* astream() ：非阻塞式，提高系统吞吐量高并发Web应用、IO密集型任务。
* batch() ：批量处理多个输入高并发场景，需要同时处理大量请求。
* abatch() ：非阻塞式，提高系统吞吐量高并发Web应用、IO密集型任务。

## 4.1 invoke() 方法
invoke() 是 LangChain 中最核心的方法，它的工作模式是阻塞式的，即程序会等待模型完全生成整个响
应后，再一次性将结果返回给用户。

简单来说， invoke 方法的作用就是：
1. 接收你的输入（问题、指令、对话历史等）
2. 发送给 LLM 模型（如 GPT-4、Llama、Claude 等）
3. 返回模型的响应（文本回复 + 元数据信息）

基本语法:

In [ ]:
response = model.invoke(
    input,        # 必填：输入内容（字符串、字典列表、消息对象列表等）
    config=None,  # 可选：运行配置（回调、标签、元数据等）
    stop=None,    # 可选：停止序列，只能用关键字传入
    **kwargs,     # 可选：本次调用的其他模型参数，如 temperature、max_tokens
)

参数解析：invoke 一共有 4 个参数，只有 `input` 是必填的：

| 参数 | 作用 | 详见 |
|---|---|---|
| `input` | 输入内容，类型：`str` / `list[dict]` / `list[Message]` / `list[tuple]` 等。**必填** | 4.1.1 |
| `config` | 运行配置（`RunnableConfig`），如回调、标签、元数据、运行名称，主要用于监控和追踪 | 4.1.2 |
| `stop` | 停止序列，模型生成到指定字符串就停止。只对本次调用生效 | 4.1.2 |
| `**kwargs` | 其他模型参数，如 `temperature`、`max_tokens`。只对本次调用生效，会覆盖初始化时的设置 | 4.1.2 |

### 4.1.1 输入参数详解 - 文本输入
invoke方法非常灵活，支持三种形式的输入： 文本输入 、 字典列表 、 消息对象列表 。

1、文本输入(最简单)简单的一次性问答，直接传入一个问题或指令。

✅适用场景：快速测试，不需要保留对话历史的简单生成任务。

❌缺点：无法设置系统提示（system prompt），无法传递对话历史

In [1]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)
# 向模型发送单条数据
response = model.invoke("翻译成英文：你好世界")
print(response)


content='Hello World' additional_kwargs={'refusal': None, 'reasoning_content': 'The user wants me to translate "你好世界" into English. This is a straightforward Chinese phrase where "你好" means "hello" and "世界" means "world." The phrase is structured as a greeting followed by its object, which works naturally in English as well. Given its simplicity, a direct translation captures both the literal meaning and the typical use of this phrase as a common programming example.'} response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 36, 'total_tokens': 119, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 80, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 36}, 'model_provider': 'deepseek', 'model_name': 'deepse

2. 字典列表(最灵活)

创建字典列表组成消息。一条消息通常包含 role（角色） 、 content（内容） 等信息。

✅适用场景：可以设置系统提示，表达多轮对话历史，JSON 兼容，易于序列化和网络传输，生产环境
推荐。

❌缺点：代码稍微多一点（但更清晰）

格式：

In [1]:
messages = [
    {"role": "system", "content": "系统提示"},
    {"role": "user", "content": "用户消息"},
    {"role": "assistant", "content": "AI回复"},  # 可选，用于对话历史
    {"role": "user", "content": "继续提问"},
]

角色说明:
1. system：系统提示，用于设置模型的行为和风格。
2. user：用户消息，包含用户的问题或指令。
3. assistant：模型回复，包含模型的回复或结果。
4. user：用户继续提问，用于继续轮对话。

In [3]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

# 使用字典格式构建消息
messages = [
    {"role":"system","content":"你是一个专业的数学老师。"},
    {"role":"user","content":"什么是斐波那契数列？"}
]
# 向模型发送单条数据
response = model.invoke(messages)
print(f"模型回复: {response.content}")


模型回复: 斐波那契数列是一个非常经典的数列。它的定义很简单：

> 从 \(0\) 和 \(1\) 开始，之后每一项都等于前两项之和。

如果记第 \(n\) 项为 \(F_n\)，常见写法是：

\[
F_0=0,\quad F_1=1
\]

\[
F_n=F_{n-1}+F_{n-2}\quad(n\ge 2)
\]

于是得到：

\[
0,1,1,2,3,5,8,13,21,34,55,89,144,\dots
\]

有时也从 \(1,1\) 开始写：

\[
1,1,2,3,5,8,13,21,\dots
\]

这两种写法本质一样，只是下标起点不同。

---

### 它为什么叫“斐波那契数列”？

它由意大利数学家莱昂纳多·斐波那契在 13 世纪研究“兔子繁殖问题”时引入西方。问题是：假设一对兔子每月生一对新兔子，新兔子出生两个月后成熟，并开始繁殖，那么每月兔子对数会怎样变化？在理想条件下，就得到了这个数列。

不过，更早的印度数学家也已经研究过类似数列。

---

### 一个重要公式：通项公式

斐波那契数列虽然是递推定义的，但它也有直接计算公式：

\[
F_n=\frac{\varphi^n-\psi^n}{\sqrt5}
\]

其中：

\[
\varphi=\frac{1+\sqrt5}{2}\approx 1.618
\]

这是著名的黄金比例；而

\[
\psi=\frac{1-\sqrt5}{2}\approx -0.618
\]

当 \(n\) 很大时，\(\psi^n\) 会趋近于 0，所以：

\[
F_n\approx \frac{\varphi^n}{\sqrt5}
\]

因此，斐波那契数列与黄金比例关系非常密切。相邻两项的比值：

\[
\frac{F_{n+1}}{F_n}
\]

会越来越接近黄金比例 \(\varphi\)。

---

### 一些常见性质

例如：

\[
F_0+F_1+\cdots+F_n=F_{n+2}-1
\]

\[
F_1^2+F_2^2+\cdots+F_n^2=F_nF_{n+1}
\]

相邻项互质：

\[
\gcd(F_n,F_{n+1})=1
\]

这些性质使斐波那契数列在数论、组合数学中很有用。

---

### 它有什么用？

斐波那契数列出现在很

表达多轮历史对话

In [4]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

# 使用字典格式构建消息
messages = [
    {"role":"system","content":"你是一个专业的数学老师。"},
    {"role":"user","content":"2 + 3 * 2 = ? "},
    {"role":"assistant","content":"8"},
    {"role":"user","content":"我刚才问了什么问题？"}
]
# 向模型发送单条数据
response = model.invoke(messages)
print(f"模型回复: {response.content}")


模型回复: 你刚才问的是：

**2 + 3 * 2 = ?**


如果不传递历史，AI 会"失忆"，只能根据当前问题回答。

In [2]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

# 使用字典格式构建消息
messages1 = [
    {"role": "system", "content": "你是一个非常友好的AI助手"},
    {"role": "user", "content": "你好，我叫小明"},
]
# 向模型发送单条数据
response = model.invoke(messages1)
print(f"模型回复: {response.content}")

messages2 = [
{"role": "user", "content": "我叫什么名字？"}
]
#第二次对话
response2 = model.invoke(messages2)
# 打印响应
print(f"AI的回复2：{response2.content}")

模型回复: 你好，小明！很高兴认识你 😊 我是你的AI助手，有什么我可以帮你的吗？无论是学习、聊天、写东西还是解决问题，都可以告诉我～


AI的回复2：我不知道你的名字。你还没有告诉我。你可以直接告诉我，我就能记住并在这次对话中使用。


3. 消息对象列表
使用内置的消息类（如 SystemMessage, HumanMessage, AIMessage），将消息对象列表输入模型。

✅适用场景：需要类型检查（针对大型项目）、IDE 自动补全的场景

❌缺点：代码较长、不如字典简洁、难以序列化（JSON）


消息类型对照：

1. SystemMessage: 系统消息，用于设置模型的行为或提供背景信息
2. HumanMessage: 用户消息，包含用户输入
3. AIMessage: 模型回复消息，包含模型生成的回复


In [7]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

# 使用字典格式构建消息
messages1 = [
    SystemMessage(content="你是一个专业的数学老师。"),
    HumanMessage(content="2 + 3 * 2 = ?"),
    AIMessage(content="8"),
    HumanMessage(content="我刚才问什么问题了?"),
]
# 向模型发送单条数据
response = model.invoke(messages1)
print(f"模型回复: {response.content}")


模型回复: 你刚才问的是：**“2 + 3 * 2 = ?”**  
答案是 **8**。


4. 补充：元组列表（简写）

除了上面三种写法，还可以用 `(角色, 内容)` 元组表示一条消息，写法最短，适合快速测试。角色可以写 `"system"`、`"human"`（或 `"user"`）、`"ai"`（或 `"assistant"`）。

In [3]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

# 使用 (角色, 内容) 元组构建消息
messages = [
    ("system", "你是一个翻译助手，把用户的话翻译成英文"),
    ("human", "今天天气真好"),
]
response = model.invoke(messages)
print(f"模型回复: {response.content}")

模型回复: The weather is really nice today.


### 4.1.2 invoke 的其他参数：config、stop 与 **kwargs

invoke 的完整签名是 `invoke(input, config=None, *, stop=None, **kwargs)`（源码见 4.1.3）。除了 `input` 和 `config`，还有 `stop` 和 `**kwargs` 两个参数。

签名中的 `*` 表示后面的参数**只能用关键字传入**：`model.invoke(问题, stop=["6"])` ✅，`model.invoke(问题, None, ["6"])` ❌。

#### 1. config：运行配置

`config` 是一个字典（类型为 `RunnableConfig`），可以设置以下键，**全部可选**：

| 键 | 作用 | 常见用途 |
|---|---|---|
| `run_name` | 本次运行的名称，默认为模型类名（如 `ChatDeepSeek`） | 在追踪记录中区分不同的调用 |
| `tags` | 标签列表 | 给调用分类，方便筛选 |
| `metadata` | 元数据字典（值要能转成 JSON） | 记录用户 ID、会话 ID 等业务信息 |
| `callbacks` | 回调处理器列表 | 在调用开始、结束时自动执行自定义逻辑，如记日志、统计耗时（见 4.1.4） |
| `configurable` | 运行时修改"可配置字段"的值 | 配合 `init_chat_model(configurable_fields=...)`，在调用时切换模型或参数 |
| `run_id` | 本次运行的唯一 ID，不传则自动生成 | 与自己系统中的日志关联 |
| `max_concurrency` | 最大并发数 | 用于 `batch()` 批量调用 |
| `recursion_limit` | 最大递归次数，默认 25 | 用于 Agent、LangGraph 等会循环调用的场景 |

注意：`run_name`、`tags`、`metadata` **不会**出现在返回的 AIMessage 中，而是记录在 LangChain 的**运行记录（Run）**里，供 LangSmith 等追踪工具使用。在本地可以用 `collect_runs()` 把运行记录收集起来查看：

In [4]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
from langchain_core.tracers.context import collect_runs

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

with collect_runs() as cb:  # 收集 with 代码块中产生的运行记录
    response = model.invoke(
        "用一句话介绍 LangChain",
        config={
            "run_name": "介绍LangChain",      # 运行名称
            "tags": ["笔记演示"],              # 标签
            "metadata": {"user_id": "u_001"},  # 元数据
        },
    )

run = cb.traced_runs[0]  # 本次调用对应的运行记录
print("回答:", response.content)
print("运行名称:", run.name)
print("标签:", run.tags)
print("元数据:", run.metadata)

回答: LangChain 是一个用于构建和编排大语言模型应用的开源框架，帮助开发者把模型、提示、工具、数据和记忆等组件串联成完整工作流。
运行名称: 介绍LangChain
标签: ['笔记演示']
元数据: {'user_id': 'u_001', 'ls_provider': 'deepseek', 'ls_model_name': 'deepseek-v4-flash', 'ls_model_type': 'chat', 'ls_temperature': None, 'ls_integration': 'langchain_chat_model', 'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2', 'langchain-openai': '1.6.4', 'langchain-deepseek': '1.1.1'}}


config 中的信息都记录在了运行记录里。元数据中除了我们传入的 `user_id`，其余以 `ls_` 开头的键（模型提供商、模型名等）和 `lc_versions` 是 LangChain 自动补充的。

#### 2. config 中的 configurable：调用时切换模型

初始化时用 `configurable_fields` 声明哪些参数允许在调用时修改，之后就可以通过 `config["configurable"]` 临时更换，不用重新创建模型：

In [5]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    configurable_fields=("model", "temperature"),  # 允许在调用时修改 model 和 temperature
)
question = "用一个字回答：天空是什么颜色？"

response1 = model.invoke(question)  # 使用默认的 deepseek-v4-flash
response2 = model.invoke(
    question,
    config={"configurable": {"model": "deepseek:deepseek-v4-pro"}},  # 本次调用换成 deepseek-v4-pro
)
print(f"默认模型：{response1.response_metadata['model_name']}，回答：{response1.content}")
print(f"切换后：{response2.response_metadata['model_name']}，回答：{response2.content}")

默认模型：deepseek-flash，回答：蓝
切换后：deepseek-v4-pro，回答：蓝


#### 3. stop 与 **kwargs：只对本次调用生效的模型参数

`stop` 和 `**kwargs` 会合并到本次请求中，**覆盖**初始化时的同名设置，但只对这一次调用生效（`02-model-init-parameters.ipynb` 第 8 节也演示过）：

In [6]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，方便观察 max_tokens 的效果
)
question = "从1数到10，用逗号分隔，只输出数字"

print("正常调用：", model.invoke(question).content)
print("stop=['6']：", model.invoke(question, stop=["6"]).content)      # 生成到 "6" 就停止
print("max_tokens=5：", model.invoke(question, max_tokens=5).content)  # 最多生成 5 个 token
print("再次正常调用：", model.invoke(question).content)               # 上面的参数只对当次调用生效

正常调用： 1,2,3,4,5,6,7,8,9,10


stop=['6']： 1,2,3,4,5,


max_tokens=5： 1,2,3


再次正常调用： 1,2,3,4,5,6,7,8,9,10


**小结**：想改变模型**怎么回答**（长度、停止词、温度等），用 `stop` / `**kwargs`；想**记录和追踪**这次调用（名称、标签、业务信息、耗时统计），用 `config`。

### 4.1.3 invoke 方法的返回值
invoke 返回一个 AIMessage对象 ，源码如下：

In [ ]:
def invoke(
    self,
    input: LanguageModelInput,
    config: RunnableConfig | None = None,
    *,
    stop: list[str] | None = None,
    **kwargs: Any,
) -> AIMessage:
    pass

下面打印具体的类型和内部包含的内容：

In [10]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_API_BASE = os.getenv("OPENAI_API_BASE")


model = init_chat_model(
    model="openai:gpt-5.6-luna",
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE
)
response = model.invoke([HumanMessage(content="2 + 3 * 2 = ?")])
print("response的类型:", type(response))
rprint(response)


response的类型: <class 'langchain_core.messages.ai.AIMessage'>


AIMessage(
    content='8',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 5,
            'prompt_tokens': 4395,
            'total_tokens': 4400,
            'completion_tokens_details': None,
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 3840,
                'image_tokens': None,
                'text_tokens': None
            }
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5.6-luna',
        'system_fingerprint': None,
        'id': 'resp_048e19a45efacd61016ab4b3d02cf087d283465fb84a3fdeb0',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0d1de-6008-75e1-8ad1-3adf94528e2e-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 4395,
        'output_tokens': 5,
        'total_tokens': 4400,
        'input_token_details': {'cache_read': 3840},
        'output_token_details': {}
    }
)

这里的rich.print 是为了打印 response 对象的详细内容，包括 content、role 等。它会自动格式化输出，使内容更易读。

不过response自身自带了pretty_print()方法，可以美化 response.content的输出

In [9]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

# 使用字典格式构建消息
messages1 = [
    SystemMessage(content="你是一个专业的数学老师。"),
    HumanMessage(content="2 + 3 * 2 = ?"),
    AIMessage(content="8"),
    HumanMessage(content="我刚才问什么问题了?"),
]
# 向模型发送单条数据
response = model.invoke(messages1)
response.pretty_print()


================================== Ai Message ==================================

你刚才问的是：**“2 + 3 * 2 = ?”**  
我当时的回答是：**8**。


不过，AIMessage 中包含丰富的信息。上面 gpt-5.6-luna 的例子没有思考过程，下面换成我们常用的 DeepSeek（默认开启思考模式），问同一个问题，看看完整的返回结构，再逐个字段说明：

In [7]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from rich import print as rprint

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)
response = model.invoke([HumanMessage(content="2 + 3 * 2 = ?")])
rprint(response)

AIMessage(
    content='8',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'We need answer basic arithmetic. Need follow order ops multiplication before 
addition: 3*2=6, +2=8. Need concise.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 31,
            'prompt_tokens': 39,
            'total_tokens': 70,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 29,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 0,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 39
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '1355de17-476e-4dca-aaf9-318de7fad8b9',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0d1f4-4f64-7cf1-96f2-a769f6e8a01e-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 39,
        'output_tokens': 31,
        'total_tokens': 70,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 29}
    }
)

**逐个字段说明**（数值来自上面这次调用）：

```python
AIMessage(
    content='8',                          # ① 回答正文，最常用的字段
    additional_kwargs={                   # ② 厂商返回的"非标准"额外字段
        'refusal': None,                  #    模型拒绝回答时的拒绝理由（OpenAI 规范字段），正常为 None
        'reasoning_content': 'We need…'   #    DeepSeek 特有：思考过程（开启思考模式时才有）
    },
    response_metadata={                   # ③ 响应元数据：基本照搬厂商 API 返回的信息
        'token_usage': {                  #    厂商原始格式的 token 用量（各厂商格式不同）
            'completion_tokens': 31,      #      输出 token 数（包含思考 token）
            'prompt_tokens': 39,          #      输入 token 数
            'total_tokens': 70,           #      总 token 数 = 输入 + 输出
            'completion_tokens_details': {            # 输出 token 明细
                'accepted_prediction_tokens': None,   #   OpenAI "预测输出"功能中被采纳的 token
                'audio_tokens': None,                 #   音频 token
                'reasoning_tokens': 29,               #   思考 token：31 个输出 token 中有 29 个用于思考
                'rejected_prediction_tokens': None,   #   "预测输出"中未被采纳的 token
                'text_tokens': None                   #   文本 token
            },
            'prompt_tokens_details': {                # 输入 token 明细
                'audio_tokens': None,                 #   音频 token
                'cache_write_tokens': None,           #   写入缓存的 token
                'cached_tokens': 0,                   #   命中缓存的 token（这部分价格更低）
                'image_tokens': None,                 #   图片 token
                'text_tokens': None                   #   文本 token
            },
            'prompt_cache_hit_tokens': 0,   #    DeepSeek 特有：命中缓存的输入 token
            'prompt_cache_miss_tokens': 39  #    DeepSeek 特有：未命中缓存的输入 token
        },
        'model_provider': 'deepseek',     #    模型提供商（LangChain 添加）
        'model_name': 'deepseek-flash',   #    实际提供服务的模型：请求的是 deepseek-v4-flash，服务端返回的是 deepseek-flash
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',  # 后端配置指纹，服务端配置变化时会改变
        'id': '1355de17-476e-4dca-aaf9-318de7fad8b9',  # 厂商生成的请求 ID，向厂商反馈问题时提供
        'finish_reason': 'stop',          #    结束原因：stop 正常结束 / length 达到 max_tokens 被截断 / tool_calls 要调用工具
        'logprobs': None                  #    token 概率，设置 logprobs=True 时才有
    },
    id='lc_run--01a0d1f4-4f64-7cf1-96f2-a769f6e8a01e-0',  # ④ LangChain 生成的消息 ID："lc_run-" + 运行 ID + 序号
    tool_calls=[],                        # ⑤ 模型请求调用的工具（学到工具调用时会用到）
    invalid_tool_calls=[],                # ⑥ 解析失败的工具调用（如参数不是合法的 JSON）
    usage_metadata={                      # ⑦ LangChain 统一格式的 token 用量（推荐使用）
        'input_tokens': 39,               #    输入 token 数
        'output_tokens': 31,              #    输出 token 数（包含思考 token）
        'total_tokens': 70,               #    总 token 数
        'input_token_details': {'cache_read': 0},  # 输入明细：cache_read 为命中缓存的 token
        'output_token_details': {'reasoning': 29}  # 输出明细：reasoning 为思考 token
    }
)
```

打印时还有两个字段没显示出来（因为是默认值）：`type='ai'`（消息类型，AIMessage 固定为 `ai`）和 `name=None`（消息发送者的名称，可选）。

**💡 这些字段是从哪来的？**

下面是 DeepSeek 接口对同一个问题返回的原始 JSON（另一次调用的结果，所以数值和上面略有不同，思考过程已截断）：

```json
{
 "id": "c0a22a2a-d914-4f53-90dd-f82811fa2d65",
 "object": "chat.completion",
 "created": 1790228646,
 "model": "deepseek-flash",
 "choices": [{
   "index": 0,
   "message": {"role": "assistant", "content": "8", "reasoning_content": "We need answer simple arithmetic order o…"},
   "logprobs": null,
   "finish_reason": "stop"
 }],
 "usage": {
  "prompt_tokens": 39, "completion_tokens": 39, "total_tokens": 78,
  "prompt_tokens_details": {"cached_tokens": 0},
  "completion_tokens_details": {"reasoning_tokens": 37},
  "prompt_cache_hit_tokens": 0, "prompt_cache_miss_tokens": 39
 },
 "system_fingerprint": "aeb56401ca74e127821c4f9126dcb669"
}
```

| 原始 JSON 中的字段 | 对应 AIMessage 中的位置 |
|---|---|
| `choices[0].message.content` | `content` |
| `choices[0].message.reasoning_content` | `additional_kwargs["reasoning_content"]` |
| `usage` | `response_metadata["token_usage"]`，同时被转换成统一格式的 `usage_metadata` |
| `model`、`id`、`system_fingerprint` | `response_metadata` 中的同名字段（`model` 改名为 `model_name`） |
| `choices[0].finish_reason`、`logprobs` | `response_metadata` 中的同名字段 |
| `created`（创建时间，精确到秒） | 被丢弃 |

反过来，原始 JSON 中没有、后来补上的字段：

- **LangChain 添加的**：`model_provider`、消息 `id`、`tool_calls`、`invalid_tool_calls`、`usage_metadata`
- **openai SDK 补充的**：`refusal`，以及 token 明细中那些值为 `None` 的字段。DeepSeek 并没有返回它们，只是 SDK 的数据结构中预留了这些位置，所以显示为 `None`

**使用建议**

1. **读取 token 用量优先用 `usage_metadata`**：各厂商的 `token_usage` 格式不同（比如只有 DeepSeek 有 `prompt_cache_hit_tokens`），`usage_metadata` 是 LangChain 统一后的格式，换模型也不用改代码。
2. **两个 id 别搞混**：`response.id` 由 LangChain 生成，用于追踪；`response_metadata["id"]` 由厂商生成，向厂商反馈问题时使用。
3. **`model_name` 是服务端实际使用的模型**，可能和请求时写的名字不同（服务端会做别名映射）。
4. **看 `finish_reason` 判断回答是否完整**：`length` 表示被 max_tokens 截断了。

> 对比上面 gpt-5.6-luna 的输出：结构相同，只是没有思考过程和 DeepSeek 特有的缓存字段，`completion_tokens_details` 为 None（接口没有返回输出明细）。另外留意它的 `prompt_tokens` 高达 4395，而问题只有几个字符。多出来的输入 token 很可能是所用的中转服务附加的系统提示，这部分同样计入用量（其中 3840 个命中了缓存）。

我们可以通过键值对的方式获取相应的对应信息

In [8]:
response = model.invoke("用一句话解释什么是 AI")
# 1. 获取回复内容
print("AI 回复:", response.content)
# 2. 获取响应元数据
metadata = response.response_metadata
print(f"使用的模型: {metadata['model_name']}")
print(f"结束原因: {metadata['finish_reason']}")
print(f"模型提供商：{metadata['model_provider']}\n")

AI 回复: AI 是让机器模拟人类智能（如学习、推理、感知和决策）来完成通常需要人类智能的任务的技术。
使用的模型: deepseek-flash
结束原因: stop
模型提供商：deepseek



In [9]:
# 3. 获取 Token 使用情况
usage = metadata.get('token_usage', {})
print(f"输入 tokens: {usage.get('prompt_tokens')}")
print(f"输出 tokens: {usage.get('completion_tokens')}")
print(f"总计 tokens: {usage.get('total_tokens')}")
# 4. 获取消息 ID
print(f"消息 ID: {response.id}")

输入 tokens: 35
输出 tokens: 96
总计 tokens: 131
消息 ID: lc_run--01a0d1f4-55b8-7600-b758-c791affbeb84-0


In [10]:
# 5. 推荐：用 LangChain 统一格式的字段读取，换成其他厂商的模型，代码也不用改
print("回答正文:", response.text)              # content 中的文本部分
print("Token 用量:", response.usage_metadata)  # 统一格式的 token 用量
for block in response.content_blocks:         # 标准化的内容块：reasoning 是思考过程，text 是回答
    print(f"[{block['type']}]", (block.get("reasoning") or block.get("text"))[:60])

回答正文: AI 是让机器模拟人类智能（如学习、推理、感知和决策）来完成通常需要人类智能的任务的技术。
Token 用量: {'input_tokens': 35, 'output_tokens': 96, 'total_tokens': 131, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 69}}
[reasoning] 我们需要回答用户中文问题：“用一句话解释什么是 AI”。需要一句话。要简洁准确。可以定义：AI是让机器模拟人类智能（如学
[text] AI 是让机器模拟人类智能（如学习、推理、感知和决策）来完成通常需要人类智能的任务的技术。


### 4.1.4 为什么 AIMessage 中没有记录回答延迟？

仔细看上面的返回结果：有 token 用量、模型名称、结束原因……唯独没有"这次调用花了多长时间"。原因有三个：

**1. AIMessage 是"一条消息"，而不是"一次调用的记录"**

AIMessage 代表对话中 AI 说的一句话，主要用途是**放回对话历史**，参与下一轮对话（就像 4.1.1 中的多轮对话示例）。4.1.1 中我们还手写过 `AIMessage(content="8")`，这条消息根本没有调用过模型，自然也谈不上延迟。耗时属于"调用过程"的信息，而不是"消息内容"，所以不放在消息里。

**2. 厂商的 API 本身就不返回耗时**

`response_metadata` 基本是把厂商返回的数据照搬过来。回看 4.1.3 中 DeepSeek 返回的原始 JSON，里面只有 token 用量、模型名、结束原因等，没有任何耗时字段（只有一个精确到秒的创建时间 `created`，LangChain 也没有保留）。

对比：本地模型框架 Ollama 的接口会返回 `total_duration`（总耗时，单位纳秒）等字段，LangChain 会原样放进 `response_metadata`。所以 response_metadata 里有没有耗时，**取决于厂商有没有返回**。

**3. LangChain 把耗时记录在"运行记录（Run）"里**

LangChain 的设计是：**消息只管内容，调用过程交给追踪系统**。每次调用模型都会生成一条运行记录，里面有开始时间、结束时间、输入输出、tags、metadata 等信息。LangSmith 平台上看到的延迟，就来自这里。

> 📌 小彩蛋：`response.id` 形如 `lc_run--<uuid>-0`，中间的 uuid 就是这次运行的 ID。它是 UUID v7 格式，前 48 位是**调用开始时刻**的毫秒时间戳。但它只记录了开始时间，没有结束时间，所以仍然算不出延迟。

下面介绍几种获取延迟的方法。

**方法 1：用 time 模块手动计时（最简单）**

In [11]:
import time
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

start = time.perf_counter()  # 记录开始时间
response = model.invoke("用一句话介绍 LangChain")
elapsed = time.perf_counter() - start  # 结束时间 - 开始时间 = 耗时

output_tokens = response.usage_metadata["output_tokens"]
print("回答:", response.content)
print(f"耗时: {elapsed:.2f} 秒，输出 {output_tokens} 个 token，平均每秒 {output_tokens / elapsed:.1f} 个 token")

回答: LangChain 是一个用于编排大语言模型、工具、数据与记忆，从而快速构建复杂 LLM 应用的开源开发框架。
耗时: 2.33 秒，输出 127 个 token，平均每秒 54.4 个 token


**方法 2：回调（callbacks）自动计时**

方法 1 每次调用都要手写计时代码。回调可以把计时逻辑写一次，通过 `config` 传入后，在**每次调用开始、结束时自动触发**（在链、Agent 内部发生的模型调用也会触发）：

In [12]:
import time
from langchain_core.callbacks import BaseCallbackHandler

class TimerCallback(BaseCallbackHandler):
    """模型调用开始、结束时自动触发，统计耗时"""

    def __init__(self):
        self.start_times = {}

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        self.start_times[run_id] = time.perf_counter()  # 用 run_id 区分不同的调用

    def on_llm_end(self, response, *, run_id, **kwargs):
        elapsed = time.perf_counter() - self.start_times.pop(run_id)
        print(f"[TimerCallback] 本次调用耗时 {elapsed:.2f} 秒")

timer = TimerCallback()
response = model.invoke("用一句话介绍 LangChain", config={"callbacks": [timer]})
print("回答:", response.content)

[TimerCallback] 本次调用耗时 1.73 秒
回答: LangChain 是一个用于构建和编排大语言模型应用的开源开发框架，提供提示、链、代理、记忆、检索和工具集成等能力。


**方法 3：读取 LangChain 的运行记录**

LangChain 自己其实记录了耗时，就在 4.1.2 中见过的运行记录里。每条运行记录都有 `start_time` 和 `end_time`：

In [13]:
from langchain_core.tracers.context import collect_runs

with collect_runs() as cb:
    response = model.invoke("用一句话介绍 LangChain")

run = cb.traced_runs[0]
print("开始时间:", run.start_time)
print("结束时间:", run.end_time)
print(f"耗时: {(run.end_time - run.start_time).total_seconds():.2f} 秒")
print("运行 ID:", run.id)
print("response.id:", response.id)  # 中间那段就是运行 ID

开始时间: 2026-09-24 05:47:31.491216+00:00
结束时间: 2026-09-24 05:47:32.870666+00:00
耗时: 1.38 秒
运行 ID: 01a0d1f4-6b23-7953-9355-fa1cdc6251a5
response.id: lc_run--01a0d1f4-6b23-7953-9355-fa1cdc6251a5-0


**方法 4：LangSmith（生产环境推荐）**

LangSmith 是 LangChain 官方的追踪平台。设置下面的环境变量后，每次调用的耗时、token 用量、输入输出都会自动上传，可以在网页上查看。需要注册账号，并且注意：输入输出内容会上传到 LangSmith 的服务器。

```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=<你的 API Key>
```

| 方法 | 优点 | 适用场景 |
|---|---|---|
| time 计时 | 最简单 | 临时测一下 |
| callbacks 回调 | 写一次，自动对每次调用生效（包括链、Agent 内部的调用） | 日志、监控 |
| 运行记录 / LangSmith | 信息最完整：时间、输入输出、token、tags、metadata | 生产环境的调试与监控 |

> 💡 补充两点：
> 1. **思考模式会明显增加耗时**：模型要先生成思考内容再回答，思考 token 越多越慢。对延迟敏感的场景可以关闭思考模式。
> 2. **聊天应用更关心"首字延迟"**（TTFT，从发出请求到收到第一个字的时间）。invoke 要等全部内容生成完才返回，测不了首字延迟，需要用后面学的 `stream()` 流式输出。